# Study 916 — Withholding Drag 🌍

**How much of an international fund's dividend is eaten before it reaches you?**

A US-domiciled fund holding Japanese, German or French shares has foreign dividend tax
withheld *at source* — before the cash ever touches the fund's NAV. On a ~3% yield a
10–15% effective rate would be 30–50 bp/yr, bigger than the fund's own fee. This study
tries to **measure** that leak from public data rather than assume it.

The tape: two legs per fund — the **total-return** close and the **price-only** close.
Their difference is the distribution the fund actually paid. Headline pair **VEA**
against an **EWJ/EWU/EWG 50/30/20** blend of the same market, 2007-07-30 → 2026-06-30
(4,760 sessions), one execution lag on the rebalance, costs one-way × NAV.

*Real numbers below are the frozen headline (`docs/results.md`, fingerprint
`ca173c825c7c`); the live cells run only the offline synthetic control. As-of 2026-06-30.*


## 1. First, a ruler that actually works

A fund's *total-return* price line includes its dividends; its *price-only* line does not. Subtract one from the other and what is left is the cash the fund handed you. Does that trick give the right answer? Compare it against the dividends the funds actually declared.

> 🔬 **For the quants:** the daily difference `r_TR − r_PX` is zero off ex-date and `D_t / P_{t−1}` on it; summing over a year is the realised distribution yield. Both legs are split-adjusted; only the first is dividend-adjusted.

In [1]:
R = {'ruler': [('VEA', 302.3, 302.2, 0.1, 66), ('IEFA', 291.3, 291.7, -0.44, 29), ('EFA', 269.3, 269.2, 0.04, 47), ('VXUS', 296.7, 297.2, -0.42, 57), ('EWJ', 147.6, 147.2, 0.38, 50), ('EWG', 232.7, 232.5, 0.16, 42), ('EWU', 373.5, 372.9, 0.6, 56)]}
print('fund   differenced   declared cash   residual')
for tk, diff, cash, resid, nex in R['ruler']:
    print('%-6s %8.1f bp %12.1f bp %9.2f bp  (%d ex-dates)'
          % (tk, diff, cash, resid, nex))
print('\nThe ruler is exact to under a basis point on all seven funds.')

fund   differenced   declared cash   residual
VEA       302.3 bp        302.2 bp      0.10 bp  (66 ex-dates)
IEFA      291.3 bp        291.7 bp     -0.44 bp  (29 ex-dates)
EFA       269.3 bp        269.2 bp      0.04 bp  (47 ex-dates)
VXUS      296.7 bp        297.2 bp     -0.42 bp  (57 ex-dates)
EWJ       147.6 bp        147.2 bp      0.38 bp  (50 ex-dates)
EWG       232.7 bp        232.5 bp      0.16 bp  (42 ex-dates)
EWU       373.5 bp        372.9 bp      0.60 bp  (56 ex-dates)

The ruler is exact to under a basis point on all seven funds.


## 2. So we can measure the income. Can we measure the *tax*?

No — and this is the whole story. The tape shows what the fund **paid you**. It never shows what the Japanese and German companies **declared** before the taxman took his cut. To see the difference you would need a benchmark that was *not* taxed — and every US-listed international ETF is taxed the same way.

The plan was to use single-country funds (EWJ, EWU, EWG) as a stand-in for the untaxed market. Here is what that comparison actually produces.

In [2]:
R = {'head': [('VEA', 4760, 302.4, 254.9, -47.5, -1.07, -0.5, -0.01, -69.3, 69.0), ('IEFA', 3436, 291.4, 257.8, -33.6, -2.77, 9.4, 0.78, -12.5, 30.6), ('EFA', 6244, 269.3, 222.9, -46.4, -4.11, -29.4, -2.61, -49.3, -10.0), ('VXUS', 3875, 296.8, 256.0, -40.8, -0.89, 4.2, 0.09, -56.3, 68.4)]}
print('fund   yield   blend   raw gap (t)      after fees (t)')
for tk, n, f, b, gap, t, adj, tadj, lo, hi in R['head']:
    print('%-6s %6.1f %7.1f %8.1f (%+.2f) %12.1f (%+.2f)'
          % (tk, f, b, gap, t, adj, tadj))
print('\nThe gap is NEGATIVE: the broad funds pay out MORE than the blend,')
print('the opposite of what a withholding leak would look like.')

fund   yield   blend   raw gap (t)      after fees (t)
VEA     302.4   254.9    -47.5 (-1.07)         -0.5 (-0.01)
IEFA    291.4   257.8    -33.6 (-2.77)          9.4 (+0.78)
EFA     269.3   222.9    -46.4 (-4.11)        -29.4 (-2.61)
VXUS    296.8   256.0    -40.8 (-0.89)          4.2 (+0.09)

The gap is NEGATIVE: the broad funds pay out MORE than the blend,
the opposite of what a withholding leak would look like.


## 3. Why the gap points the wrong way

Two dull reasons, neither of them tax:

1. **Fees.** The single-country funds charge **50 bp**; VEA charges 3 bp. Distributions are paid *after* fees, so the expensive benchmark hands back less income by construction — a **47 bp** head start.
2. **Composition.** Japan yields 148 bp, Germany 233 bp, the UK 374 bp. Which three you pick, and in what proportion, moves the answer more than any tax could.

Add the fee difference back and the headline gap is **-0.5 bp/yr** — a flat zero (*t* = -0.01). Change the country weights and it swings by **72 bp**.

In [3]:
R = {'wsweep': [('EAFE-ish 50/30/20', 254.9, -47.5, -0.5, -1.07), ('equal 1/3 each', 271.5, -30.9, 16.1, -0.65), ('Japan-heavy 70/20/10', 226.5, -75.9, -28.9, -1.82), ('UK-heavy 30/50/20', 298.1, -4.3, 42.7, -0.09)], 'wsweep_span': 71.6}
print('weights                blend yield   raw gap   after fees')
for name, b, gap, adj, t in R['wsweep']:
    print('%-22s %9.1f %10.1f %11.1f' % (name, b, gap, adj))
print('\nspan of the fee-adjusted estimate: %.0f bp/yr -- the assumption,'
      % R['wsweep_span'])
print('not the tape, is doing the talking.')

weights                blend yield   raw gap   after fees
EAFE-ish 50/30/20          254.9      -47.5        -0.5
equal 1/3 each             271.5      -30.9        16.1
Japan-heavy 70/20/10       226.5      -75.9       -28.9
UK-heavy 30/50/20          298.1       -4.3        42.7

span of the fee-adjusted estimate: 72 bp/yr -- the assumption,
not the tape, is doing the talking.


## 4. What we *can* say — an inference, clearly labelled

VEA's measured **net** distribution yield is **302 bp/yr**. If you are willing to *assume* an effective withholding rate, the gross dividend and the drag follow by arithmetic. That is all this is — arithmetic on an assumption, not a measurement.

> 🔬 **For the quants:** gross = net / (1 − w), drag = net · w / (1 − w). The tape identifies `net` and nothing else.

In [4]:
R = {'infer': [(0.05, 318, 16), (0.1, 336, 34), (0.12, 344, 41), (0.15, 356, 53), (0.2, 378, 76), (0.25, 403, 101)], 'central_w': 0.12}
print('assumed rate   implied gross   implied drag')
for w, gross, drag in R['infer']:
    star = '   <- central assumption' if abs(w - R['central_w']) < 1e-9 else ''
    print('   %3.0f%%          %5.0f bp       %5.0f bp/yr%s' % (w*100, gross, drag, star))
print('\nHonest range: 16 to 101 bp/yr. A 6x spread, set entirely by a')
print('number the tape cannot see.')

assumed rate   implied gross   implied drag
     5%            318 bp          16 bp/yr
    10%            336 bp          34 bp/yr
    12%            344 bp          41 bp/yr   <- central assumption
    15%            356 bp          53 bp/yr
    20%            378 bp          76 bp/yr
    25%            403 bp         101 bp/yr

Honest range: 16 to 101 bp/yr. A 6x spread, set entirely by a
number the tape cannot see.


## 5. And could you do anything about it anyway?

No. Suppose the drag really is 41 bp/yr. Swapping the cheap broad fund for the single-country blend does not avoid it — those funds are US-domiciled too, taxed identically — and it costs about **1 percentage point a year** in fees and composition (excess-of-cash return difference -103 bp/yr, HAC *t* = -1.06, essentially unchanged from 0 to 25 bps of cost).

The one lever that *is* real lives on your tax return, not in the market: a US **taxable** account can usually claim the foreign tax credit; an IRA or 401(k) cannot. Which is an argument about *where* you hold the fund, not *which* fund you hold.

## 6. Is the detector broken, or is the answer really null?

Fair question. We build a fake world where the broad fund genuinely does lose an extra slice of its dividend to tax, and check the estimator finds it — then a world where nobody is taxed differently, and check it stays quiet. This cell runs live, offline.

In [5]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from withholding import data, strategy as st
tr, px, truth = data.synthetic_panel(signal_strength=1.0, seed=916)
planted = st.synthetic_detect(tr, px, truth, n_boot=400)
tr0, px0, truth0 = data.synthetic_panel(signal_strength=0.0, seed=916)
null = st.synthetic_detect(tr0, px0, truth0, n_boot=400)
print('planted a %.1f bp leak -> estimator reads %+.1f bp (t %+.2f)'
      % (planted['true_gap_bp'], planted['measured_gap_bp'], planted['t_hac']))
print('planted NO leak       -> estimator reads %+.1f bp, CI [%+.1f, %+.1f]'
      % (null['measured_gap_bp'], null['ci_low_bp'], null['ci_high_bp']))
print('\nThe detector works. The real-tape null is about the benchmark,')
print('not about the machinery.')

planted a 25.6 bp leak -> estimator reads +25.4 bp (t +9.17)
planted NO leak       -> estimator reads +0.1 bp, CI [-0.1, +0.4]

The detector works. The real-tape null is about the benchmark,
not about the machinery.


## Verdict

- **Signal — None.** The question is not answerable from the tape. The measurement is exact (the ruler reproduces declared cash to under a basis point on all seven funds), but the *gross* dividend never appears in any price series, and every candidate benchmark is a US fund taxed the same way. The headline gap is **-47.5 bp/yr with the wrong sign** (HAC *t* = -1.07), collapses to **-0.5 bp** once fees are added back, is insignificant in both eras and flips sign between them, and swings 72 bp across plausible country weights.
- **Tradability — Mirage.** Nothing to bank. The 'gross' benchmark is taxed identically and costs ~103 bp/yr more to own. The only genuine lever is the foreign tax credit — a tax-return mechanic, off-tape.
- **What survives.** The ruler. Total return minus price return measures any fund's realised income yield to within a basis point, and the desk keeps it.